In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import os 
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# PLSSVD: fixed-dimension generalization to held-out trials
Five shuffled folds, with each trial held out exactly once (approximately 80% train / 20% test per fold). Trial assignment is stratified by subject and condition. There are no test A/B halves and no component tuning. All folds use the same participants, pairing and source selection.

Average trials within each condition and partition, then average conditions with equal weight. Fit and test on these condition-averaged time courses.

```bash
python -u plssvd_eval.py --root /path/to/iEEGvsMEG \
  --meg-kind full_concatenated --n-components 5 --n-splits 5 \
  --output-dir /path/to/new_plssvd_eval_kfold
```


In [ ]:
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
sys.path.insert(0, str(ROOT.parent / 'LB'))
from plssvd_eval_utils import (ValidationOptions, 
                               prepare_trial_cache, load_trial_cache,
                               validate_plssvd, plot_plssvd_validation, 
                               load_plssvd_results,)

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
MEG_KIND = 'full_concatenated'
# Illustrative batch defaults; this notebook loads results rather than running the analysis.
options = ValidationOptions(
    n_components=5, repeats=5,
    n_null=199, seed=2026, split_unit='trial', block_scaling='none',
)
CACHE_DIR = ROOT / 'out' / 'trial_cache'
OUTPUT_DIR = ROOT / 'out' / 'plssvd_eval_kfold'


## Trial cache and fold assignment
The cache contains unaveraged trials. For five folds, every subject/condition needs at least five trials. Within each subject and condition, shuffle once and divide into five nearly equal chunks. Fold f tests on chunk f and trains on the other four. Test chunks are disjoint and exhaust all trials.

Optional metadata columns are `modality`, `subject`, `condition`, `trial_index` (within condition), and `split_group`. With `split_unit='group'`, whole groups are held out together across conditions; at least five distinct groups and nonempty conditions in every fold are required. Unequal group sizes can give unequal test sizes. `permutation_block` is retained for compatibility with other workflows; this condition-averaged evaluation does not use a condition-label null.

The cache configuration does not depend on MEG_KIND: reuse the same trial cache when changing dataset construction. Changed input files, cache preprocessing configuration or metadata require a new cache directory.


In [ ]:
result = load_plssvd_results(OUTPUT_DIR)
if result['validation_options'].get('schema_version', 1) < 3:
    raise ValueError('These results predate disjoint folds and condition averaging. Run the updated batch script into a new directory.')
MEG_KIND = result['validation_options']['meg_kind']
print(f"Loaded {MEG_KIND}: {result['validation_options']['repeats']} disjoint test folds, "
      f"fixed k={result['validation_options']['n_components']}")
display(pd.Series(result['validation_options'], name='Saved evaluation settings'))
display(result['trial_counts'] if not result['trial_counts'].empty else result['participants'])


## Fit and test on averaged responses
For each subject, separately in train and test:
1. Average selected trials within each condition.
2. Apply the existing preprocessing using training-estimated parameters.
3. Average condition responses with equal weight (not weighted by trial counts).

Dataset construction preserves the requested representation:
- iEEG: concatenate every subject's electrode features, with no cross-subject channel alignment or averaging.
- MEG `full_concatenated`: concatenate all subjects' source features.
- MEG `full_average`: average subjects by existing source index (matching shapes required).
- MEG `coverage_average`: average subjects after sampling each at electrode coverage.
- MEG `paired_coverage` / `random_control`: concatenate the paired sampled features; preserve fixed participant/source assignments across folds.

The resulting matrices are **time × features**. The sample-space PCA/SVD factors used by PLSSVD receive these averaged matrices, not individual trials or stacked conditions. Five components remain fixed and configurable. All fitted weights, centering and prediction maps use training data only. Upstream iEEG preprocessing may already have pooled trials; this evaluation does not undo that upstream operation.

`split_audit` records only train/test membership. Each trial appears once as test and four times as training with five folds. Training sets overlap; test sets do not.


In [ ]:
display(result['summary'].round(3))
display(result['metric_summary'].query("partition == 'test'").round(3))

## Goodness on held-out data: signal fit and shared covariance

**1. Own-modality reconstruction.** For each modality, project the test signals onto the training-learned PLS weight space and reconstruct them using the same weights:
Any training whole-modality scale is undone for reconstruction. `ieeg_reconstruction_fraction` and `meg_reconstruction_fraction` report how much of each test signal the learned space retains.

**2. Cross-modal prediction.** Training ridge regressions map MEG scores to iEEG features and iEEG scores to MEG features. `predict_ieeg_q2` and `predict_meg_q2` are `1 − test squared error / training-mean-baseline squared error`. Positive Q² improves on that baseline; negative Q² is worse.

**3. Held-out cross-covariance.** Let C be the full test-feature cross-covariance after partition centering and the training-learned scaling, and let Wx/Wy be the training PLS weights. The test score covariance matrix is `Wx.T @ C @ Wy`. `crosscov_energy_fraction` is its squared Frobenius norm divided by `||C||F²`.This is the fraction of observed test cross-covariance energy retained by the trained spaces.

Also report `mean_paired_covariance` (signed diagonal mean in native score units), per-component train/test covariance, `paired_crosscov_energy_fraction` (diagonal energy only), and signed paired score Pearson r. A covariance fraction can be high even when the absolute shared signal is weak, so inspect these measures together. Covariance retention relative to training is in `summary.csv` and can exceed one or become negative.

In [ ]:
#display(result['components'].round(3))
plot_plssvd_validation(result)


## Stability across folds

`metric_summary.csv` reports mean, SD, median and range of each train/test metric. The performance-consistency plot shows every test split. These ranges are descriptive, not population confidence intervals.

`fold_stability.csv` compares the refitted score time courses between every pair of splits, separately in iEEG and MEG. It uses mean absolute Pearson correlation with one-to-one component matching, handling sign flips and changes in order. `fold_component_pairs.csv` records the assignments. This post-hoc matching describes stability only and never changes test prediction or cross-covariance metrics. Components may rotate across nearly degenerate solutions; simple matching is not rotation invariant.
Test trial sets are disjoint, but training sets overlap; folds are not independent replicates.


In [ ]:
# Descriptive consistency across refitted folds with disjoint test trials and overlapping training trials.
display(result['fold_stability'].query("partition == 'test'").round(3))
display(result['metric_summary'].query("partition == 'test'").round(3))

## Null diagnostics for condition-averaged responses
Only fold 0 supplies these optional diagnostics; the model and preprocessing remain fixed.

1. **Temporal shift:** circularly shift the MEG condition-averaged test score time courses; statistic is mean signed paired component Pearson r.
2. **Spatial correspondence:** shuffle MEG forward-pattern rows within iEEG subject/region groups at electrode correspondences; statistic is mean absolute paired component pattern correlation.

These are conditional surrogate/correspondence diagnostics, not automatically calibrated significance tests. Condition-label and condition-contrast tests are omitted because the main analysis averages conditions. Fold consistency is reported separately.


In [ ]:
display(result['null_tests'])
print(f'Results folder: {OUTPUT_DIR}')